# AD mouse brain

This notebook runs the packaged `admouse` workflow from data preparation
through downstream analysis. Edit the paths in **Setup**, then enable the run
switches for the steps you need. The saved outputs below come from the packaged
preset and do not require the external dataset.

## Setup

In [1]:
from pathlib import Path

import pandas as pd

from CytoBridge.workflow import (
    WorkflowOptions,
    build_workflow_plan,
    load_workflow_config,
    render_workflow_plan,
    run_workflow,
)

PRESET = 'admouse'
RAW_H5AD = Path("data/admouse_raw.h5ad")
OUTPUT_DIR = Path("tutorial_outputs/admouse")
ALIGNED_H5AD = OUTPUT_DIR / "preprocess" / 'admouse_aligned.h5ad'
MODEL_DIR = OUTPUT_DIR / "training"


RUN_PREPARATION = False
RUN_PREPROCESS_AND_TRAIN = False
RUN_DOWNSTREAM = False

In [2]:
config, preset_source = load_workflow_config(PRESET)
dataset = config["dataset"]
scientific = config["scientific"]
downstream = config["downstream"]

pd.DataFrame(
    {
        "setting": [
            "dataset",
            "preset",
            "raw time column",
            "cell annotation",
            "observed training times",
            "classifier neighbors",
        ],
        "value": [
            dataset["display_name"],
            preset_source,
            config["preprocess"]["time_key"],
            dataset["annotation_key"],
            ", ".join(map(str, downstream["observed"])),
            scientific["classifier_k"],
        ],
    }
)

,setting,value
0,dataset,AD mouse brain
1,preset,packaged preset: admouse
2,raw time column,Timepoint
3,cell annotation,major_annotation
4,observed training times,"0.0, 1.0, 2.0"
5,classifier neighbors,1


## Data preparation

The preset records the count layer, time mapping, spatial coordinates, and
alignment settings used for this dataset. The plan below shows the input and
output paths before any long-running work starts.

In [3]:
preparation_options = WorkflowOptions(
    input_h5ad=RAW_H5AD,
    output_dir=OUTPUT_DIR,
    steps=("preprocess",),
)
preparation_plan = build_workflow_plan(
    config,
    source=preset_source,
    options=preparation_options,
)
print(render_workflow_plan(preparation_plan))

CytoBridge workflow plan
dataset: AD mouse brain (admouse)
config: packaged preset: admouse
scientific parameters: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=1
steps:
  preprocess: ready (GPU recommended for spatial alignment)
    output: tutorial_outputs/admouse/preprocess/admouse_aligned.h5ad
    note: The AD raw-H5AD workflow fits a learned edge predictor from the seven complete ligand-receptor pairs represented by the targeted panel and uses the validation-selected threshold 0.9956824779510498. Broader CCI analyses require a more complete panel. The packaged all-spatial configuration is the corresponding no-LR-prior ablation.
    edge predictor: not requested during preprocessing
  train: skipped; add --train to run (GPU required for production training)
  downstream: skipped (GPU recommended)


In [4]:
if RUN_PREPARATION:
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Update RAW_H5AD before preprocessing: {RAW_H5AD}")
    preparation_result = run_workflow(config, options=preparation_options)
    preparation_result
else:
    print("Data preparation is off. Set RUN_PREPARATION = True to run it.")

Data preparation is off. Set RUN_PREPARATION = True to run it.


## Training

The full training run starts from the raw H5AD, writes the aligned data, fits
the interaction edge predictor when the preset requires one, and trains the
CytoBridge model. A production run requires a CUDA-capable environment.

In [5]:
training_options = WorkflowOptions(
    input_h5ad=RAW_H5AD,
    output_dir=OUTPUT_DIR,
    steps=("preprocess", "train"),
    train=True,
)
training_plan = build_workflow_plan(
    config,
    source=preset_source,
    options=training_options,
)
print(render_workflow_plan(training_plan))

CytoBridge workflow plan
dataset: AD mouse brain (admouse)
config: packaged preset: admouse
scientific parameters: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=1
steps:
  preprocess: ready (GPU recommended for spatial alignment)
    output: tutorial_outputs/admouse/preprocess/admouse_aligned.h5ad
    note: The AD raw-H5AD workflow fits a learned edge predictor from the seven complete ligand-receptor pairs represented by the targeted panel and uses the validation-selected threshold 0.9956824779510498. Broader CCI analyses require a more complete panel. The packaged all-spatial configuration is the corresponding no-LR-prior ablation.
    edge predictor: will be trained automatically
      graph database: package: CytoBridge/workflow_databases/CellChatDB.ligrec.mouse.csv
      database source: bundled formal CellChatDB resource
      interaction cutoff: 0.012106042891492197
      decision threshold source: validation-selected during de novo training
      output: tutorial_

In [6]:
if RUN_PREPROCESS_AND_TRAIN:
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Update RAW_H5AD before training: {RAW_H5AD}")
    training_result = run_workflow(config, options=training_options)
    training_result
else:
    print("Training is off. Set RUN_PREPROCESS_AND_TRAIN = True to start it.")

Training is off. Set RUN_PREPROCESS_AND_TRAIN = True to start it.


## Downstream analysis

Downstream analysis uses the aligned H5AD and fitted model from the training
directory. The dataset preset supplies the interpolation times, classifier
settings, trajectory simulation, growth analysis, and ligand–receptor options.

In [7]:
downstream_options = WorkflowOptions(
    aligned_h5ad=ALIGNED_H5AD,
    model_dir=MODEL_DIR,
    output_dir=OUTPUT_DIR / "downstream",
    steps=("downstream",),
)
downstream_plan = build_workflow_plan(
    config,
    source=preset_source,
    options=downstream_options,
)
print(render_workflow_plan(downstream_plan))

CytoBridge workflow plan
dataset: AD mouse brain (admouse)
config: packaged preset: admouse
scientific parameters: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=1
steps:
  preprocess: skipped (GPU for spatial alignment)
  train: skipped; add --train to run (GPU required for production training)
  downstream: ready (GPU recommended for SDE simulation and classifier fitting)
    model format: current
    output: tutorial_outputs/admouse/downstream/downstream
    simulation: observed=[0.0, 1.0, 2.0], interpolated=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.1, 2.2, 2.3, 2.4, 2.5], initial particles=all observed t0 cells, dt=0.01, sigma=0.03, daughter noise=0, growth alpha=1, trajectory mode=global_t0_extrapolation
      scope: Generated states belong to one split-SDE rollout initialized from t0; later states are global extrapolations rather than independently observed-anchored interval interpolations.
    interpolation and cla

In [8]:
if RUN_DOWNSTREAM:
    missing = [path for path in (ALIGNED_H5AD, MODEL_DIR) if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing trained artifacts: {missing}")
    downstream_result = run_workflow(config, options=downstream_options)
    downstream_result
else:
    print("Downstream analysis is off. Set RUN_DOWNSTREAM = True to run it.")

Downstream analysis is off. Set RUN_DOWNSTREAM = True to run it.


## Paper figures

The figure notebooks load the corresponding packaged result tables and save
the paper PDFs and PNGs:

- [Interaction-prior ablation](../paper_figures/lr_prior_ablation_stvcr.ipynb)
- [Five-dataset benchmark](../paper_figures/loto_benchmark.ipynb)
- [Training histories](../paper_figures/training_histories.ipynb)

## Saved files

In [9]:
pd.DataFrame(
    {
        "file or directory": [
            "aligned data",
            "training directory",
            "downstream directory",
        ],
        "path": [
            str(ALIGNED_H5AD),
            str(MODEL_DIR),
            str(OUTPUT_DIR / "downstream"),
        ],
    }
)

,file or directory,path
0,aligned data,tutorial_outputs/admouse/preprocess/admouse_al...
1,training directory,tutorial_outputs/admouse/training
2,downstream directory,tutorial_outputs/admouse/downstream
